# QML 강의 - 3일차

## MediQ-GAN fine-tuning과 QGAN 비교

- 목표: QGAN(생성자 전체가 양자)과 MediQ-GAN(생성자 안의 일부만 양자)을 같은 지표로 비교함
- MediQ-GAN 양자 stream은 원 논문 spec(16큐빗 × sub-generator 5개, depth 8). pretrained checkpoint에서 출발해 20 step만 fine-tuning함
- 16×16 축소판이라 생성 이미지는 논문 수준에 못 미침. 5번 섹션 끝에서 논문 결과와 비교함
- 전체 학습은 생략함: 16큐빗 시뮬레이션이 비싸 25 epoch에 196분(로컬 CPU)
- 개념 설명은 3일차 설명 자료(`practice/day3_slides/`)에 있음

| 섹션 | 내용 | 소요(Colab 무료 CPU) |
|---|---|---|
| 1 | 2일차 복습: 실습 자산 로드, QGAN 생성자 재정의 | 1 ~ 2분 |
| 2 | QGAN baseline: 600 step checkpoint로 생성 + 지표 4종 | 수 초 |
| 3 | MediQ-GAN 구조: dual-stream 생성자 정의, shape·파라미터·회로 규모 확인 | 십여 초 |
| 4 | 라이브 fine-tuning: 10 epoch checkpoint에서 20 step | 약 14분 |
| 5 | 두 모델 비교: 같은 지표, 논문 결과와의 차이 | 수 초 |
| 6 | 정리 | - |

## 0. Google Colab 사용법

2일차와 같음. Colab이 처음이면 2일차 노트북의 0번 섹션을 먼저 봄.

- 노트북을 연 뒤 `파일 → Drive에 사본 저장`. "이 노트북은 Google에서 작성하지 않았습니다" 경고가 뜨면 `무시하고 계속`. 런타임은 기본 CPU
- 2일차 런타임은 남아 있지 않으므로 1번 섹션부터 순서대로 실행함(건너뛰면 `NameError`)
- 대부분 수 초 안에 끝남. 오래 걸리는 셀: 3번 구조 확인(십여 초), 4번 fine-tuning 20 step(로컬 CPU 약 7분, Colab 무료 CPU 실측 약 14분)
- 연결이 끊기면 다시 연결해 1번 섹션부터 실행함(fine-tuning 셀을 빼면 1 ~ 2분)
- 오류는 메시지 마지막 줄부터 확인함

## 1. 2일차 복습 + 실습 자산 로드

2일차 데이터·checkpoint를 저장소에서 받아 씀. 2일차에 직접 만든 결과는 쓰지 않음(수업마다 같은 출발점에서 비교하기 위함).

| 파일(`practice/assets/`) | 내용 | 사용 위치 |
|---|---|---|
| `df_preprocessed.npy` | DF 115장, 16×16, 0 ~ 1 (2일차 segmentation + `INTER_AREA` 축소) | 실제 이미지, 판별자 학습 |
| `nv_preprocessed.npy` | NV 345장, 같은 전처리 | 클래스 평균 비교용(이 노트북에서는 로드만 함) |
| `qgan_df_16px.pt` | QGAN 600 step checkpoint (2일차 수업은 400 step) | 2번 섹션 baseline |
| `mediqgan_e010.pt` | MediQ-GAN 10 epoch (280 step) checkpoint | 4번 섹션 fine-tuning 출발점 |
| `mediqgan_final.pt` | MediQ-GAN 25 epoch (700 step) checkpoint | 4·5번 섹션 비교 |

checkpoint는 모두 `generator_state_dict`, `discriminator_state_dict`, `config`(구조·학습 설정), `history`(step별 손실)를 가짐.

In [ ]:
!pip install -q pennylane pennylane-lightning koreanize-matplotlib  # lightning: 3일차 16큐빗 회로용 C++ 시뮬레이터
import os, sys, math, time, subprocess
import numpy as np, torch, torch.nn as nn
import pennylane as qml
import matplotlib.pyplot as plt
# 그래프 한글 표시 (설정하지 않으면 글자가 □로 깨짐)
try:
    import koreanize_matplotlib  # 나눔고딕 폰트 등록 + 기본 폰트 지정
except Exception as e:
    print("한글 폰트 설정 실패(그래프 한글만 깨짐, 진행에는 지장 없음):", e)

print(f"PennyLane {qml.__version__}, PyTorch {torch.__version__}")

In [ ]:
# 실습 자산(전처리 데이터 + checkpoint)은 저장소 practice/assets/에 있음
REPO_URL = "https://github.com/hyunhp/skin_disease_qml.git"
REPO_DIR = "/content/skin_disease_qml"  # Colab 기본 작업 폴더 아래에 clone

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"], check=False)  # 재실행 시 최신 자산으로 갱신
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=True)  # 최신 commit만 받아 수 MB

ASSET_DIR = f"{REPO_DIR}/practice/assets"
sys.path.insert(0, f"{REPO_DIR}/practice")  # gen_metrics import 경로
import gen_metrics as gm  # 2일차와 같은 지표 모듈 (to_unit_scale, evaluate, comparison_table 등)

for need in ["qgan_df_16px.pt", "mediqgan_e010.pt", "mediqgan_final.pt", "df_preprocessed.npy", "nv_preprocessed.npy"]:
    if not os.path.exists(f"{ASSET_DIR}/{need}"):
        raise RuntimeError(f"실습 자산 {need}이 없음 - git clone 출력의 오류 메시지를 확인하고 이 셀을 다시 실행할 것")
print(f"실습 자산 경로: {ASSET_DIR}")
print(f"포함 항목: {sorted(os.listdir(ASSET_DIR))}")

df_imgs = np.load(f"{ASSET_DIR}/df_preprocessed.npy")  # (115, 16, 16), 0 ~ 1
nv_imgs = np.load(f"{ASSET_DIR}/nv_preprocessed.npy")  # (345, 16, 16), 클래스 평균 비교용
IMG_SIZE = df_imgs.shape[1]                             # 16 (2일차와 같음)
print(f"2일차 전처리 데이터 로드: DF {df_imgs.shape}, NV {nv_imgs.shape}, IMG_SIZE={IMG_SIZE}")
print(f"DF 픽셀 범위 {df_imgs.min():.2f} ~ {df_imgs.max():.2f}, 0.05 미만(배경) 비율 {(df_imgs < 0.05).mean():.0%}")

# (선택) 2일차에 Drive에 저장한 본인 checkpoint를 쓰려면 주석 해제
# from google.colab import drive; drive.mount('/content/drive')

### QGAN 생성자 재정의

`state_dict`는 파라미터 값만 저장하므로, checkpoint를 불러오려면 같은 구조의 클래스가 먼저 있어야 함. 2일차 4·5번 섹션의 코드와 같음.

- 구조: sub-generator 4개 × 7큐빗(데이터 6 + ancilla 1), depth 10
- 출력: 회로마다 ancilla qubit = 0인 확률 64개(2⁶)만 patch로 씀(post-selection) → 4 patch를 이어붙여 256픽셀

In [ ]:
class PatchQuantumGenerator(nn.Module):
    """2일차와 같은 patch 방식 생성자 - sub-generator N개가 patch를 하나씩 만들어 이어붙임.

    인자 기본값은 아래에서 checkpoint config로 항상 덮어쓰므로 의미 없음."""
    def __init__(self, n_generators=4, n_data_qubits=4, n_ancillas=1, q_depth=3):
        super().__init__()
        self.n_generators = n_generators
        self.n_data_qubits = n_data_qubits
        self.n_qubits = n_data_qubits + n_ancillas  # 6 + 1 = 7
        self.q_depth = q_depth
        self.patch_dim = 2 ** n_data_qubits         # patch 하나의 픽셀 수 = 64
        self.anc_stride = 2 ** n_ancillas           # ancilla qubit=0 항목의 간격 ([::2])

        dev = qml.device("default.qubit", wires=self.n_qubits)  # backprop 미분용
        n_qubits = self.n_qubits

        @qml.qnode(dev, interface="torch", diff_method="backprop")
        def circuit(noise, weights):                 # noise (7,), weights (depth, 7)
            for i in range(n_qubits):
                qml.RY(noise[i], wires=i)            # noise encoding (학습 대상 아님)
            for layer in range(q_depth):
                for i in range(n_qubits):
                    qml.RY(weights[layer, i], wires=i)  # 학습 파라미터 θ
                for i in range(n_qubits - 1):
                    qml.CZ(wires=[i, i + 1])         # 인접 큐빗 얽힘
            # 전체 측정 → forward에서 ancilla qubit=0 항목만 사용 (post-selection)
            return qml.probs(wires=range(n_qubits))  # (128,) = 2^7개 확률

        self.circuit = circuit
        # sub-generator마다 독립된 회전각 (depth × 큐빗) = 10 × 7 = 70개씩
        self.q_params = nn.ParameterList([
            nn.Parameter(torch.rand(q_depth, self.n_qubits) * math.pi) for _ in range(n_generators)
        ])

    def forward(self, batch_size, device="cpu"):
        all_patches = []
        for gen_idx in range(self.n_generators):
            weights = self.q_params[gen_idx]
            # ancilla qubit=0 항목만 선택 → patch 밝기 총합(0 ~ 1)도 학습됨
            batch_patches = [self.circuit(torch.rand(self.n_qubits, device=device) * math.pi, weights)[::self.anc_stride]
                             for _ in range(batch_size)]  # 샘플마다 새 noise(0 ~ π)로 회로 1회 실행, 각 (64,)
            all_patches.append(torch.stack(batch_patches))  # (batch, 64)
        return torch.cat(all_patches, dim=1).float()     # (batch, 4 × 64) = (batch, 256)

    @property
    def output_dim(self):
        return self.n_generators * self.patch_dim

print("PatchQuantumGenerator 정의 완료 (학습 전 랜덤 상태)")

## 2. QGAN 생성 + 지표 4종

600 step checkpoint로 이미지를 생성하고 2일차 지표 4종으로 평가함. 이 값이 MediQ-GAN과 비교할 baseline임.

| 지표 | 확인하는 것 | 좋은 방향 |
|---|---|---|
| ① 픽셀 강도 분포 | 밝기 분포가 실제와 비슷한가 | 실제와 겹침 |
| ② 클래스 평균 상관 | DF 평균 이미지의 형태를 재현했는가 | 높을수록 |
| ③ 최근접 실제 거리 | 학습 이미지를 베꼈는가 | 0이 아니면 됨(0이면 암기) |
| ④ 샘플 간 거리 | 생성 이미지끼리 다른가 | 0이 아니면 됨(0이면 mode collapse) |

정의와 코드는 `practice/gen_metrics.py`, 설명은 2일차 노트북 6번 섹션.

In [ ]:
ckpt_path = f"{ASSET_DIR}/qgan_df_{IMG_SIZE}px.pt"
checkpoint = torch.load(ckpt_path, map_location="cpu")  # GPU에서 저장했어도 CPU로 불러옴
cfg = checkpoint["config"]  # 학습 당시 구조 설정 (sub-generator 수, 큐빗 수, depth)
print(f"불러온 checkpoint 설정: {cfg}")

# 저장된 q_params 모양과 맞도록 config로 같은 구조를 다시 만듦
gen = PatchQuantumGenerator(n_generators=cfg["n_generators"], n_data_qubits=cfg["n_data_qubits"],
                            n_ancillas=cfg["n_ancillas"], q_depth=cfg["q_depth"])
gen.load_state_dict(checkpoint["generator_state_dict"])  # 구조가 다르면 여기서 오류
gen.eval()  # 평가 모드 (학습하지 않음)

hist = checkpoint["history"]  # step별 손실 기록
print(f"학습된 생성자 로드 (2일차와 같은 구조, {len(hist['loss_g'])} step)")
print(f"마지막 20 step 평균 손실: loss_g {np.mean(hist['loss_g'][-20:]):.3f}, loss_d {np.mean(hist['loss_d'][-20:]):.3f}")

In [ ]:
# 평가 sample 수: QGAN 64장, MediQ-GAN 16장 (16큐빗 시뮬레이션이 비쌈)
# 8 ~ 16장이면 상관계수가 0.05 ~ 0.17까지 흔들림 - 소수점보다 격차를 볼 것
N_EVAL_QGAN = 64
N_EVAL = 16
with torch.no_grad():  # 생성만 하므로 gradient 계산 생략
    qgan_fake = gen(batch_size=N_EVAL_QGAN).numpy().reshape(N_EVAL_QGAN, IMG_SIZE, IMG_SIZE)  # (64, 16, 16), 확률 scale

qgan_u = gm.to_unit_scale(qgan_fake, "prob")  # 확률 scale → 이미지별 min-max로 0 ~ 1
real_u = gm.to_unit_scale(df_imgs, "unit")    # 이미 0 ~ 1 - 그대로
print(f"QGAN 출력: 장당 픽셀 합 평균 {qgan_fake.reshape(N_EVAL_QGAN, -1).sum(1).mean():.2f} (patch 4개 × 0 ~ 1)")

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for j in range(4):
    axes[0, j].imshow(qgan_u[j], cmap="gray"); axes[0, j].set_title(f"QGAN 생성 {j}", fontsize=9); axes[0, j].axis("off")
    axes[1, j].imshow(real_u[j], cmap="gray"); axes[1, j].set_title(f"실제 DF {j}", fontsize=9); axes[1, j].axis("off")
plt.suptitle("QGAN 생성 이미지(위) vs 실제 DF(아래)")
plt.tight_layout(); plt.show()

In [ ]:
# 지표 4종 (5번 섹션에서 MediQ-GAN과 비교)
qgan_scores = gm.evaluate(qgan_fake, df_imgs, "prob")  # scale 변환 후 지표 계산까지 한 번에
for k, v in qgan_scores.items():
    print(f"  {k}: {v:.4f}")

### QGAN baseline 읽기

참고값(README 결과표, 로컬 CPU): ② 0.195 ± 0.012(64장 × 10시드), ③ 1.64, ④ 1.92. 실행마다 noise가 달라 소수점 둘째 자리부터 바뀜.

- 이미지: 밝은 점이 흩어질 뿐 병변 덩어리가 생기지 않음. patch 4개를 서로 따로 만들어 patch 사이 연결을 학습하지 못함
- ② 0.2 수준: 평균 형태(가운데가 밝음)를 일부만 재현함
- ③④는 이미지별 min-max로 늘린 값이라 실제 DF의 값과 직접 비교하지 않음(5번 섹션)
- 실제 DF 중에도 segmentation이 모서리 테두리만 남긴 이미지가 있음(2일차 segmentation 한계)

## 3. MediQ-GAN 구조

layer 단위 hybrid: 생성자 안에서 고전 stream과 양자 stream이 병렬로 돌고, concat 후 decoder로 합쳐짐. 공식 저장소(github.com/QingyueJ-nd/MediQ-GAN)와 같은 방식으로 구현함.

| 단계 | 내용 |
|---|---|
| Feature split | latent vector 256차원을 절반씩 고전 stream(MLP)과 양자 stream에 나눔 |
| 양자 stream | 128차원을 16 site로 보고, site [0, 4, 8, 11, 15]만 sub-generator 회로 5개가 하나씩 처리(나머지 11칸은 0) |
| 회로 | RY → RX encoding, [학습 RY + 인접 CZ] × 8, 큐빗별 PauliX 기댓값의 평균 → site당 스칼라 1개 |
| Fusion | 1×1 conv로 16채널 projection → 고전 stream 출력과 concat → decoder → Tanh |

batch 크기 B 기준 tensor shape:

```
z (B, 256)
 ├─ z[:, :128] ─ classical_stream: Linear 128→32 → ReLU → Linear 32→128 ───────────── (B, 128) ─┐
 └─ z[:, 128:] ─ view (B, 16 site, 8) ─ q_align: Linear 8→16 ─ (B, 16, 16)                       │
                 site 5개만 회로 실행: (16,) → 기댓값 16개 → 평균 → 스칼라                         │
                 4×4 맵에 배치 (B, 5, 4, 4) ─ proj_qexp: 1×1 conv 5→16 ─ (B, 16, 4, 4) → (B, 256) ┤
                                                                         concat ─ (B, 384) ◄────┘
                                          decoder: Linear 384→32 → ReLU → Linear 32→256 → Tanh ─ (B, 256) → 16×16
```

- 원 논문과 다른 점: 고전 encoder/decoder가 conv 대신 MLP, 입력이 64×64 RGB 대신 16×16 grayscale, 판별자가 WGAN-GP critic 대신 표준 BCE 판별자
- 양자 stream(16큐빗 × 5, depth 8)은 그대로임. depth 8은 논문 그림 기준(공식 저장소 CLI 기본값은 6)
- QGAN과의 차이: QGAN은 회로 출력(확률)이 곧 픽셀이지만, MediQ-GAN은 회로 출력이 decoder의 입력 특징 중 일부임

In [ ]:
class ToyDualStreamGenerator(nn.Module):
    """MediQ-GAN dual-stream 생성자 (구조와 shape는 위 마크다운 참고)."""

    def __init__(self, in_dim=64, n_qubits=16, q_depth=8, n_generators=5, out_dim=64,
                 q_ch=16, dev_name="lightning.qubit"):
        super().__init__()
        assert in_dim % 2 == 0, "고전/양자 stream으로 절반씩 나누려면 in_dim이 짝수여야 함"
        self.half = in_dim // 2                    # 256 → stream당 128
        assert self.half % 16 == 0, "양자 쪽 절반을 4×4=16 site로 나누려면 in_dim/2가 16의 배수여야 함"
        self.site_dim = self.half // 16            # site당 8차원
        self.n_qubits = n_qubits
        self.q_depth = q_depth
        self.n_generators = n_generators
        # 16 site 중 회로가 처리할 site를 고르게 고름 (공식 저장소와 같은 방식)
        self.site_idx = torch.linspace(0, 15, steps=n_generators).round().long().tolist()  # [0, 4, 8, 11, 15]

        # 고전 stream (원 논문 conv encoder를 MLP로 단순화): (B, 128) → (B, 128)
        self.classical_stream = nn.Sequential(
            nn.Linear(self.half, 32), nn.ReLU(),
            nn.Linear(32, self.half),
        )

        # site 벡터(8차원)를 큐빗 수(16)에 맞춤 (공식 저장소의 q_align): 회로 입력 각도 16개
        self.q_align = nn.Linear(self.site_dim, n_qubits)

        dev = qml.device(dev_name, wires=n_qubits)  # 16큐빗 상태벡터 = 2^16 = 65,536 복소수

        # lightning.qubit은 backprop 미지원 → adjoint 미분 (forward 1회 + backward 1회 분량으로 gradient 전체를 구함)
        @qml.qnode(dev, interface="torch", diff_method="adjoint")
        def q_circuit(x, weights):                 # x (16,) 입력 각도, weights (8, 16) 학습 파라미터
            for i in range(n_qubits):
                qml.RY(x[i], wires=i)              # 같은 값으로 RY → RX encoding
                qml.RX(x[i], wires=i)
            for layer in range(q_depth):
                for i in range(n_qubits):
                    qml.RY(weights[layer, i], wires=i)  # 학습 파라미터 θ
                for i in range(n_qubits - 1):
                    qml.CZ(wires=[i, i + 1])       # 인접 큐빗 얽힘 (사슬 형태)
            return [qml.expval(qml.PauliX(i)) for i in range(n_qubits)]  # 큐빗별 기댓값 16개 (-1 ~ 1)

        self.q_circuit = q_circuit
        self.q_weights = nn.ParameterList([
            nn.Parameter(torch.rand(q_depth, n_qubits) * math.pi) for _ in range(n_generators)
        ])  # sub-generator 5개 × (8 × 16) = 640개
        # 5채널 4×4 맵 → q_ch채널 (1×1 conv, 공식 저장소의 proj_qexp): 위치마다 5개 값을 16개로 섞음
        self.proj_qexp = nn.Conv2d(n_generators, q_ch, 1)

        # fusion 특징 → 최종 이미지 (원 논문 conv decoder를 MLP로 단순화)
        fused_dim = self.half + q_ch * 16          # 128 + 256 = 384
        self.decoder = nn.Sequential(
            nn.Linear(fused_dim, 32), nn.ReLU(),
            nn.Linear(32, out_dim), nn.Tanh(),     # 출력 -1 ~ 1
        )

    def quantum_stream(self, z_quantum):
        # 회로 5 × B회 실행 - 가장 느린 부분
        B = z_quantum.shape[0]
        sites = z_quantum.view(B, 16, self.site_dim)  # (B, 16 site, 8)
        tokens = self.q_align(sites)                   # (B, 16 site, 16 큐빗) - 회로 입력 각도
        q_maps = torch.zeros(B, self.n_generators, 4, 4, dtype=z_quantum.dtype)  # (B, 5, 4, 4), 선택되지 않은 11칸은 0
        for g, si in enumerate(self.site_idx):         # sub-generator g ↔ site si
            outs = torch.stack([torch.stack(self.q_circuit(tokens[b, si], self.q_weights[g]))
                                for b in range(B)])    # (B, 16) - 샘플마다 회로 1회
            y, x = divmod(si, 4)                       # site 번호 → 4×4 맵 좌표
            q_maps[:, g, y, x] = outs.mean(dim=1).to(z_quantum.dtype)  # 큐빗 평균 → 스칼라 (B,)
        return self.proj_qexp(q_maps).flatten(1)       # (B, 16, 4, 4) → (B, 256)

    def forward(self, z):
        # latent vector를 절반씩 두 stream에 분배 (논문 Fig.1 "Feature Split")
        z_classical, z_quantum = z[:, :self.half], z[:, self.half:]  # 각 (B, 128)
        classical_feat = self.classical_stream(z_classical)          # (B, 128)
        quantum_feat = self.quantum_stream(z_quantum)                # (B, 256)
        fused = torch.cat([classical_feat, quantum_feat], dim=1)     # (B, 384) - skip-connection fusion
        return self.decoder(fused)                                   # (B, 256), -1 ~ 1

print("ToyDualStreamGenerator 정의 완료")

### 구조 확인: shape, 파라미터 수, 회로 규모

학습 전 랜덤 가중치로 forward 1회를 실행해 shape를 확인하고, 모듈별 파라미터 수와 회로 하나의 gate 수를 셈.

In [ ]:
IN_DIM = IMG_SIZE * IMG_SIZE  # 256 = latent vector 차원 = 출력 픽셀 수
toy_gen = ToyDualStreamGenerator(in_dim=IN_DIM, n_qubits=16, q_depth=8, n_generators=5, out_dim=IN_DIM)

z = torch.randn(1, IN_DIM)  # 구조 확인용 latent vector 1개, (1, 256)
t0 = time.time()
with torch.no_grad():
    demo_out = toy_gen(z)   # 16큐빗 회로 5회 실행
print(f"dual-stream 출력 shape: {tuple(demo_out.shape)}, 값 범위 {demo_out.min():.2f} ~ {demo_out.max():.2f} "
      f"(forward {time.time()-t0:.1f}초, 학습 전 랜덤 가중치)")

# 모듈별 파라미터 수 (설명 자료의 파라미터 표와 같은 값)
total = sum(p.numel() for p in toy_gen.parameters())
for name, m in toy_gen.named_children():
    n = sum(p.numel() for p in m.parameters())
    print(f"  {name:17s} {n:6,d}  ({n / total:5.1%})")
print(f"  {'합계':15s} {total:6,d}")

In [ ]:
# 회로 하나의 규모: gate 수와 depth (sub-generator 5개가 같은 구조)
specs = qml.specs(toy_gen.q_circuit)(torch.rand(16), toy_gen.q_weights[0])
print(specs)

# 상태벡터 크기 비교: 2일차 QGAN 7큐빗 vs MediQ-GAN 16큐빗
for nq in (7, 16):
    print(f"{nq:2d}큐빗: 상태벡터 {2**nq:6,d}차원 (complex128 {2**nq * 16 / 1024:7.1f} KB)")

### 구조 확인 결과 읽기

- 파라미터: 30,000개 중 양자 회전각은 640개(2.1%). 2일차 QGAN 생성자는 280개(4 × 10 × 7) 전부가 양자 회전각임
- 시간: 파라미터는 2%지만 forward·backward 시간 대부분은 16큐빗 회로 시뮬레이션에서 나옴
- 회로 하나: RY 144(encoding 16 + 학습 128), RX 16, CZ 120(15 × 8) = 280 gate. sub-generator 5개라 샘플 하나에 회로 5회

<details>
<summary>확인 질문 1. 양자 stream 출력을 0으로 바꿔도 decoder는 이미지를 만들 수 있는가?</summary>

만들 수 있음. decoder 입력 384차원 중 128차원은 고전 stream에서 오고, proj_qexp의 bias도 있음. 양자 stream이 생성 결과에 얼마나 기여하는지는 이 구조만으로 알 수 없고, 양자 stream을 고전 layer로 바꾼 ablation으로 확인해야 함(이 실습에서는 하지 않음).
</details>

<details>
<summary>확인 질문 2. 회로마다 큐빗 16개의 기댓값을 평균해 스칼라 1개로 줄임. 이 설계의 비용은?</summary>

회로 1회 실행으로 얻는 정보가 16개 값에서 1개로 줄어듦. 공식 저장소의 설계를 그대로 따른 것이며, 대신 proj_qexp(1×1 conv)가 site 5개의 스칼라를 16채널로 늘려 decoder에 넘김.
</details>

## 4. 라이브 fine-tuning

10 epoch checkpoint(`mediqgan_e010.pt`)에서 20 step을 더 학습함. 수렴한 모델에서 시작하면 손실이 거의 움직이지 않음.

- 소요 시간: 1 step 로컬 CPU 약 20초, Colab 약 40초(20 step 실측 13.8분)
- 결과: 20 step은 출발점까지의 학습(10 epoch, 280 step)의 7%라 이미지·지표가 거의 그대로임(Colab 실행 예: 상관 0.027 → -0.010). 학습된 모습은 25 epoch 최종본으로 봄
- 판별자: QGAN과 같은 Table 4 구조. checkpoint의 판별자 상태도 함께 불러옴
- scale: 출력이 Tanh(-1 ~ 1)라 실제 이미지를 `2x - 1`로 맞춤. 평가 때 `to_unit_scale(x, "tanh")`로 0 ~ 1 복원

In [ ]:
# 10 epoch checkpoint에서 출발 (덜 학습돼야 변화가 보임)
mq_ckpt = torch.load(f"{ASSET_DIR}/mediqgan_e010.pt", map_location="cpu")
mq_cfg = mq_ckpt["config"]  # 구조·학습률·latent 분포·batch 등 학습 당시 설정
print(f"MediQ-GAN checkpoint 설정: {mq_cfg}")
print(f"학습 epoch {mq_ckpt['epoch']}, 기록된 step {len(mq_ckpt['history']['loss_g'])}")

# config로 같은 구조를 만든 뒤 가중치를 덮어씀
mq_gen = ToyDualStreamGenerator(in_dim=IN_DIM, n_qubits=mq_cfg["n_qubits"], q_depth=mq_cfg["q_depth"],
                                n_generators=mq_cfg["n_generators"], out_dim=IN_DIM)
mq_gen.load_state_dict(mq_ckpt["generator_state_dict"])

In [ ]:
class ClassicalDiscriminator(nn.Module):
    """QGAN과 동일한 논문 Table 4 판별자 (5층 FC)."""
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),  # (B, 256) → (B, 64)
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 8), nn.ReLU(),
            nn.Linear(8, 1), nn.Sigmoid(),        # (B, 1), 진짜일 확률 (0 ~ 1)
        )
    def forward(self, x):
        return self.net(x)

mq_disc = ClassicalDiscriminator(IN_DIM)
mq_disc.load_state_dict(mq_ckpt["discriminator_state_dict"])  # 판별자도 같은 시점으로
print(f"판별자 파라미터 {sum(p.numel() for p in mq_disc.parameters()):,}개")

In [ ]:
# latent vector 분포는 checkpoint 설정을 따름 (학습 때와 다른 분포를 넣으면 decoder가 본 적 없는 입력이 됨)
Z_DIST = mq_cfg.get("z_dist", "normal")

def sample_z(n):
    return torch.rand(n, IN_DIM) if Z_DIST == "uniform" else torch.randn(n, IN_DIM)  # (n, 256)

print(f"latent vector 분포: {Z_DIST}")

# 전후 비교용으로 fine-tuning 전 결과 저장
with torch.no_grad():
    before = mq_gen(sample_z(N_EVAL)).numpy().reshape(N_EVAL, IMG_SIZE, IMG_SIZE)  # (16, 16, 16), -1 ~ 1
print(f"fine-tuning 전 생성 완료 - 값 범위 {before.min():.3f} ~ {before.max():.3f} (Tanh이므로 -1 ~ 1이 정상)")

### 학습 1 step의 계산 과정

2일차 QGAN 학습 루프와 같은 순서임. 차이는 생성자 출력 scale(Tanh)과 미분 방식(adjoint)뿐임.

| 순서 | 코드 | shape / 내용 |
|---|---|---|
| ① 실제 batch | `real_tanh[torch.randint(...)]` | (4, 256), 115장에서 sampling with replacement |
| ② 판별자 손실 | `bce(D(real), 1) + bce(D(G(z).detach()), 0)` | `detach`로 생성자 쪽 gradient 차단 |
| ③ 판별자 갱신 | `loss_d.backward(); opt_d.step()` | 판별자 파라미터만 바뀜 |
| ④ 생성자 손실 | `bce(D(G(z)), 1)` = −log D(G(z)) | 새 z로 다시 생성 (회로 5 × 4 = 20회) |
| ⑤ 생성자 갱신 | `loss_g.backward(); opt_g.step()` | 판별자 → decoder → 양자 stream(adjoint) → q_align까지 gradient 전달 |

로그 읽는 법(2일차와 같음):

| 값 | 의미 |
|---|---|
| `loss_d` ≈ 2 ln 2 ≈ 1.386 | 판별자가 진짜와 가짜를 구분하지 못함(경쟁 유지) |
| `loss_g` ≈ ln 2 ≈ 0.693 | 판별자가 생성 이미지를 반반으로 판정 |
| `loss_d` → 0, `loss_g` 증가 | 판별자가 이기는 중(README "학습 안정성") |

강사 pretraining 기록에서 출발점 직전(10 epoch) 평균은 `loss_g` 0.98, `loss_d` 1.22, 다음 epoch(11) 평균은 1.08, 1.16임. 20 step 동안 `loss_g`가 조금 오르고 `loss_d`가 조금 내리면 pretraining과 같은 흐름임(로컬 CPU 실행 예: `loss_g` 1.02 → 1.06, `loss_d` 1.19 → 1.17).

In [ ]:
# 실제 이미지를 Tanh scale로 (0 ~ 1 -> -1 ~ 1)
real_tanh = torch.tensor(df_imgs.reshape(len(df_imgs), -1) * 2 - 1, dtype=torch.float32)  # (115, 256)

bce = nn.BCELoss()  # 판별자 출력(확률) vs 라벨(1/0)
# beta1 0.5 (GAN 관례, 기본값 0.9는 불안정)
_betas = (mq_cfg.get("beta1", 0.5), 0.999)
opt_g = torch.optim.Adam(mq_gen.parameters(), lr=mq_cfg["lr_g"], betas=_betas)   # 양자 회전각 포함 30,000개
opt_d = torch.optim.Adam(mq_disc.parameters(), lr=mq_cfg["lr_d"], betas=_betas)  # lr은 둘 다 0.0002 (checkpoint 설정)
# optimizer 상태(Adam moment)는 checkpoint에 없어 새로 시작함 - 첫 몇 step은 갱신 폭이 원래 학습과 다를 수 있음

FINE_TUNE_STEPS = 20   # 1 step: 로컬 CPU 약 20초, Colab 무료 CPU 약 40초(20 step 실측 13.8분)
                       # 오래 걸리면 값을 줄이거나 ■로 중단해도 됨 - 중단 시점까지 갱신된 가중치로 아래 셀이 실행됨
BATCH = mq_cfg.get("batch_size", 4)  # checkpoint 학습 때와 같은 batch
ft_g, ft_d = [], []    # step별 손실 기록 (아래 손실 그래프용)

mq_gen.train(); mq_disc.train()
t0 = time.time()
for step in range(FINE_TUNE_STEPS):
    # ① 실제 DF에서 BATCH장 무작위 추출 (sampling with replacement)
    batch = real_tanh[torch.randint(0, len(real_tanh), (BATCH,))]  # (4, 256)
    real_lbl, fake_lbl = torch.ones(BATCH, 1), torch.zeros(BATCH, 1)  # (4, 1)

    # ②③ 판별자: 실제→1, 생성→0 (detach로 생성자 gradient 제외)
    opt_d.zero_grad()
    fake_d = mq_gen(sample_z(BATCH)).detach()                       # (4, 256), 회로 20회
    loss_d = bce(mq_disc(batch), real_lbl) + bce(mq_disc(fake_d), fake_lbl)
    loss_d.backward(); opt_d.step()

    # ④⑤ 생성자: 생성 이미지를 1로 판정받도록 학습 (양자 stream까지 backpropagation)
    opt_g.zero_grad()
    loss_g = bce(mq_disc(mq_gen(sample_z(BATCH))), real_lbl)        # = −log D(G(z))
    loss_g.backward(); opt_g.step()                                  # adjoint로 회로 gradient 계산 - step 시간의 대부분

    ft_g.append(loss_g.item()); ft_d.append(loss_d.item())
    el = time.time() - t0  # 경과 시간으로 남은 시간 추정
    print(f"step {step+1}/{FINE_TUNE_STEPS}  loss_g={loss_g.item():.4f}  loss_d={loss_d.item():.4f}  "
          f"경과 {el/60:.1f}분  남은 예상 {(FINE_TUNE_STEPS-step-1)*el/(step+1)/60:.1f}분", flush=True)

mq_gen.eval(); mq_disc.eval()
print(f"\nfine-tuning 완료 - 총 {(time.time()-t0)/60:.1f}분, 양자 stream 파라미터까지 함께 갱신됨")

### 결과 확인: 출발점, fine-tuning 후, 25 epoch 최종본

아래 두 셀은 이미지 비교와 손실 기록을 봄. 25 epoch 최종본(`mediqgan_final.pt`)은 같은 설정으로 강사가 미리 학습한 것(로컬 CPU 196분)이며 5번 섹션 비교에도 씀.

In [ ]:
# fine-tuning 후 생성
with torch.no_grad():
    after = mq_gen(sample_z(N_EVAL)).numpy().reshape(N_EVAL, IMG_SIZE, IMG_SIZE)  # (16, 16, 16)

# 25 epoch 최종본 (README 결과표 수치)
final_ckpt = torch.load(f"{ASSET_DIR}/mediqgan_final.pt", map_location="cpu")
mq_final = ToyDualStreamGenerator(in_dim=IN_DIM, n_qubits=mq_cfg["n_qubits"], q_depth=mq_cfg["q_depth"],
                                  n_generators=mq_cfg["n_generators"], out_dim=IN_DIM)
mq_final.load_state_dict(final_ckpt["generator_state_dict"])
mq_final.eval()
with torch.no_grad():
    final = mq_final(sample_z(N_EVAL)).numpy().reshape(N_EVAL, IMG_SIZE, IMG_SIZE)  # (16, 16, 16)

# 행마다 (이름, 0 ~ 1로 되돌린 이미지)
rows = [("10 epoch(출발점)", gm.to_unit_scale(before, "tanh")),
        (f"+{FINE_TUNE_STEPS} step(오늘)", gm.to_unit_scale(after, "tanh")),
        ("25 epoch(강사 최종)", gm.to_unit_scale(final, "tanh"))]
fig, axes = plt.subplots(3, 4, figsize=(10, 7.5))
for r, (name, imgs) in enumerate(rows):
    for j in range(4):  # 행마다 앞 4장, 같은 밝기 범위(0 ~ 1)로 표시
        axes[r, j].imshow(imgs[j], cmap="gray", vmin=0, vmax=1); axes[r, j].axis("off")
    axes[r, 0].set_title(name, loc="left", fontsize=10)
plt.suptitle("MediQ-GAN - 학습이 진행되며 생성 결과가 어떻게 변하는가")
plt.tight_layout(); plt.show()

In [ ]:
# 강사 pretraining 700 step(25 epoch × 28 step) 손실 + 오늘 fine-tuning 20 step
h = final_ckpt["history"]
START = len(mq_ckpt["history"]["loss_g"])  # 280 = 10 epoch 지점 (오늘의 출발점)

def smooth(x, k=14):  # 반 epoch 이동평균 - batch 4라 step별 손실의 흔들림이 큼
    return np.convolve(x, np.ones(k) / k, mode="valid")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
for key, ax, ref in [("loss_g", axes[0], math.log(2)), ("loss_d", axes[1], 2 * math.log(2))]:
    ax.plot(h[key], color="lightgray", lw=0.8, label="강사 pretraining (step별)")
    ax.plot(np.arange(13, 13 + len(smooth(h[key]))), smooth(h[key]), color="gray", label="강사 pretraining (이동평균)")
    ft = ft_g if key == "loss_g" else ft_d
    ax.plot(np.arange(START, START + len(ft)), ft, color="crimson", lw=1.5, label=f"오늘 fine-tuning {len(ft)} step")
    ax.axvline(START, color="crimson", ls=":", lw=0.8)
    ax.axhline(ref, color="black", ls="--", lw=0.8, label=f"기준값 {ref:.3f}")
    ax.set_xlabel("step"); ax.set_title(key); ax.legend(fontsize=8)
plt.suptitle("MediQ-GAN 손실 - 점선(step 280)이 10 epoch checkpoint")
plt.tight_layout(); plt.show()

print(f"강사 학습 epoch당 평균 {np.mean(h['epoch_sec'])/60:.1f}분, 25 epoch 합계 {sum(h['epoch_sec'])/60:.0f}분 (로컬 CPU)")

### fine-tuning 결과 읽기

- 10 epoch, +20 step: 노이즈에 가깝고 손실도 거의 그대로임(Colab 실행 예: loss_d 1.19 → 1.17)
- 손실 기록(epoch 평균, `loss_g` / `loss_d`): 세 구간으로 나뉨

  | 구간 | 값 | 상태 |
  |---|---|---|
  | 1 ~ 10 epoch | 0.83 / 1.39 → 0.98 / 1.22 | 완만하게 판별자 쪽으로 기움. 오늘의 출발점 |
  | 11 ~ 21 epoch | 최대 3.37 / 0.74 (19 epoch) | 판별자 우세. 생성 이미지를 대부분 가짜로 판정 |
  | 22 epoch | 1.90 / 1.09 | 몇 step 사이에 두 손실이 급변 |
  | 23 ~ 25 epoch | 0.72 ~ 0.77 / 1.31 ~ 1.36 | 기준값 근처로 돌아감. 손실로 환산하면 판별자가 진짜·가짜 모두 약 0.5로 판정 |

- 23 epoch 이후 손실이 기준값 근처에 머물고, 최종본이 mode collapse(④ 0.30)임. 생성자가 판별자를 속이는 한 가지 출력으로 수렴한 것으로 보임. 확인한 것은 손실 기록과 최종 지표뿐이며 epoch별 이미지는 저장하지 않음
- 25 epoch: 네 장이 거의 같은 무늬임(샘플 간 거리 0.30, 실제 DF 2.505). 병변 덩어리는 보이지 않고, ② 0.37은 밝기 배치가 평균 이미지와 약하게 맞는 수준임
- 배경이 검지 않음: 실제 DF는 대부분 0, 생성 이미지는 0.2 부근(5번 섹션 histogram)
- 원인으로 보이는 단순화: 작은 MLP decoder(hidden 32), 표준 BCE 판별자, 짧은 pretraining(25 epoch). 양자 stream은 원 논문 spec 그대로임

<details>
<summary>확인 질문 3. 손실이 기준값 근처에 머무는데도 이미지가 좋아지지 않는 이유는?</summary>

GAN 손실은 판별자와 생성자의 상대적인 균형만 나타냄. 판별자가 약하면 품질이 낮은 이미지에도 손실이 기준값 근처에 머묾. 그래서 품질은 5번 섹션의 지표로 따로 평가함.
</details>

<details>
<summary>확인 질문 4. 25 epoch 최종본의 네 장이 거의 같은데 손실 곡선만으로 이를 알 수 있는가?</summary>

단정할 수 없음. 이 기록에서는 22 epoch의 급변이 단서지만, 23 ~ 25 epoch의 손실 값 자체는 균형 상태(기준값 근처)와 구분되지 않음. mode collapse는 생성 이미지끼리의 거리(지표 ④)로 확인함.
</details>

## 5. QGAN vs MediQ-GAN - 같은 지표로 비교

두 모델의 출력을 모두 0 ~ 1로 되돌린 뒤 같은 지표로 평가함(QGAN은 확률, MediQ-GAN은 -1 ~ 1). 실행마다 새 난수를 쓰므로 소수점보다 두 모델의 격차를 봄.

- 변환 방식 차이: QGAN은 이미지별 min-max, MediQ-GAN과 실제 DF는 고정 변환. ②는 영향이 없음. ③④는 MediQ-GAN만 실제 값(다양성 2.505)과 직접 비교할 수 있고, QGAN은 MediQ-GAN과의 순서만 봄
- 세 대상을 모두 이미지별 min-max로 맞춰 다시 계산해도 순서는 같음(다양성 실제 5.02, QGAN 1.83, MediQ-GAN 0.48, 로컬 CPU)

In [ ]:
# 네 생성 결과를 같은 지표로 한 표에 모음 (mode에 맞춰 0 ~ 1로 되돌린 뒤 계산)
table = gm.comparison_table({
    "QGAN (모델 단위 hybrid)": (qgan_fake, "prob"),
    "MediQ-GAN 10 epoch": (before, "tanh"),
    f"MediQ-GAN +{FINE_TUNE_STEPS} step": (after, "tanh"),
    "MediQ-GAN 25 epoch(최종)": (final, "tanh"),
}, df_imgs)
print("=== 생성품질 지표 비교 (모두 0 ~ 1로 되돌린 뒤 계산) ===")
display(table) if "display" in dir() else print(table)  # Colab·Jupyter는 표로, 그 외는 텍스트로

# 비교 기준: 실제 DF끼리의 샘플 간 거리 (④의 상한에 해당)
print(f"\n실제 DF 115장의 샘플 간 거리 평균: {gm.sample_diversity(real_u)['pairwise_mean']:.3f}")

### 비교표 읽기

README 결과표(로컬 CPU) 기준값. 오늘 실행한 값은 난수에 따라 조금 다름.

| | ② 상관 ↑ | ③ 최근접 실제 거리 | ④ 샘플 간 거리 |
|---|---|---|---|
| QGAN 600 step | 0.195 | 1.64 | 1.92 |
| MediQ-GAN 10 epoch | 0.018 | - | - |
| MediQ-GAN 25 epoch | 0.365 | 3.44 | 0.30 |
| 실제 DF | - | - | 2.505 |

- ②: 평균 형태는 MediQ-GAN 25 epoch가 더 가까움. 10 epoch(+20 step)는 0 근처로, 아직 형태가 없음
- ③: MediQ-GAN은 실제 이미지 어느 것과도 멂. 배경이 0.2 부근이라 배경이 0인 실제 이미지와 전체적으로 차이가 남. 생성 이미지가 서로 비슷해 최근접 실제 이미지도 대부분 같은 한 장(고리 모양 DF)으로 잡힘(아래 그림)
- ④: MediQ-GAN은 실제의 약 1/8로 비슷한 그림을 반복함. ②만 보면 MediQ-GAN이 낫다고 결론 내리게 됨
- `픽셀평균(원 scale)`: QGAN은 확률(patch별 합)이라 값이 작고, MediQ-GAN은 -1 ~ 1이라 음수일 수 있음. scale이 달라 모델 간 비교에 쓰지 않음

In [ ]:
# 지표 ② 상관과 ① 픽셀 분포를 그림으로
df_mean = gm.class_mean_image(real_u)  # DF의 평균 이미지 (16, 16)
models = [("QGAN", qgan_fake, "prob"), ("MediQ-GAN 25 epoch", final, "tanh")]
labels, corrs = [], []
for name, arr, mode in models:
    u = gm.to_unit_scale(arr, mode)
    labels.append(name); corrs.append(gm.mean_image_correlation(u, df_mean).mean())  # 이미지별 상관의 평균

fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
axes[0].imshow(df_mean, cmap="gray"); axes[0].set_title("DF 평균 이미지 (② 기준)"); axes[0].axis("off")
axes[1].bar(labels, corrs); axes[1].axhline(0, color="gray", lw=0.8)
axes[1].set_title("② DF 평균 이미지와의 상관계수"); axes[1].set_ylabel("상관계수")
gm.plot_pixel_histogram(real_u, gm.to_unit_scale(qgan_fake, "prob"), ax=axes[2],
                        label_real="실제 DF", label_fake="QGAN")
axes[2].set_title("① QGAN 픽셀 분포")
gm.plot_pixel_histogram(real_u, gm.to_unit_scale(final, "tanh"), ax=axes[3],
                        label_real="실제 DF", label_fake="MediQ-GAN")
axes[3].set_title("① MediQ-GAN 픽셀 분포")
plt.tight_layout(); plt.show()

In [ ]:
# 지표 ③ 최근접 실제 거리, ④ 샘플 간 거리
for name, arr, mode in models:
    u = gm.to_unit_scale(arr, mode)
    d, _ = gm.nearest_neighbor_distance(u, real_u)  # 생성 이미지별 가장 가까운 실제 이미지까지 L2 거리
    div = gm.sample_diversity(u)                    # 생성 이미지 쌍의 L2 거리 통계
    print(f"{name}: 최근접 실제거리 평균 {d.mean():.3f} (최솟값 {d.min():.3f}) / "
          f"샘플간 거리 평균 {div['pairwise_mean']:.3f} (최솟값 {div['pairwise_min']:.3f})")
print("최근접거리가 0에 가까우면 암기, 샘플간 거리가 0에 가까우면 mode collapse임")

# MediQ-GAN 생성(위)과 가장 가까운 실제 이미지(아래) - 위 행이 서로 비슷하면 mode collapse
fig, _ = gm.plot_nearest_neighbors(gm.to_unit_scale(final, "tanh"), real_u, IMG_SIZE, n_show=6)
fig.suptitle("MediQ-GAN 25 epoch: 생성(위) vs 최근접 실제 DF(아래)", y=1.02)
plt.show()

<details>
<summary>확인 질문 5. ②가 높은데 ④가 낮은 모델은 어떤 이미지를 만드는가?</summary>

평균 이미지와 비슷한 그림을 거의 그대로 반복함. ②는 이미지마다 평균 이미지와의 상관을 계산해 평균내므로, 모든 출력이 평균 이미지와 같아도 높게 나옴. MediQ-GAN 25 epoch가 이 경우임.
</details>

<details>
<summary>확인 질문 6. MediQ-GAN의 ③이 QGAN보다 큰 것은 학습 이미지를 덜 베꼈다는 뜻인가?</summary>

그렇게 해석하지 않음. ③이 0에 가까우면 암기를 의심하지만, 큰 값은 단지 실제 이미지와 다르다는 뜻임. MediQ-GAN은 배경이 0.2 부근이라 모든 실제 이미지와 거리가 멂. 또 QGAN은 이미지별 min-max로 늘린 값이라 scale도 다름.
</details>

### 실습 결과와 원 논문의 생성 이미지

이 실습의 두 모델은 병변 모양을 제대로 만들지 못함. 아래는 같은 구조를 원 논문 조건으로 학습한 결과임(저자가 고른 샘플).

| | 이 실습 | 원 논문 |
|---|---|---|
| QGAN | 16×16 grayscale, sub-generator 4개 × 7큐빗, 600 step (CPU 약 22분) | sub-generator 16개 × 8큐빗, 2,000 epoch, GPU |
| MediQ-GAN | 16×16 grayscale, MLP encoder/decoder, BCE 판별자, 25 epoch | 64×64 RGB, conv encoder/decoder, WGAN-GP critic, GPU 학습 |

**HAM10000 QGAN (Andra et al.) - Fig. 21**

<img src="https://raw.githubusercontent.com/hyunhp/skin_disease_qml/main/practice/docs/image/paper_qgan_fig21_samples.png" width="520">

(A) 원본 이미지, (B) segmentation 이미지로 학습한 QGAN의 생성 결과. (B)가 2일차와 같은 조건이며 병변이 한 덩어리로 나옴.
출처: Andra et al., Research Square 프리프린트 rs-5191116v1, Fig. 21 (CC BY 4.0)

**MediQ-GAN (Jiao et al.) - Fig. 2**

<img src="https://raw.githubusercontent.com/hyunhp/skin_disease_qml/main/practice/docs/image/paper_mediqgan_fig2_samples.png" width="760">

행은 모델, 열은 클래스. 아래에서 두 번째 행이 MediQ-GAN, 마지막 행이 실제 이미지. ISIC 2019 df 열이 이 실습의 DF와 같은 클래스임.
출처: Jiao et al., MediQ-GAN, arXiv:2506.21015v2, Fig. 2 (CC BY 4.0)

### MediQ-GAN 원 논문 결과 (논문 수치 인용)

| 데이터셋 | 내용 | 결과 |
|---|---|---|
| ISIC 2019 | 피부경검사 8클래스 | 소수 클래스(df, vasc, scc)에서 모든 고전 baseline보다 낮은 FID |
| ODIR-5k | 안저사진 | 증강 후 분류: 두 backbone(EfficientNet-B0, ViT-small) 모두 최고 성능 |
| RetinaMNIST | downsampled 안저사진 | 5개 클래스 중 4개에서 최저(최고품질) FID |

- 세 데이터셋 모두 클래스별 FID가 최고 또는 차상위(Table 2). ODIR-5k 행만 FID가 아니라 증강 후 분류 성능(Table 1)임
- 비교 대상: DCGAN, WGAN-GP, Structure-Preserving GAN, StyleGAN2-ADA, FastGAN, DiT, Stable Diffusion 3(LoRA fine-tuning), 이전 세대 hybrid 모델 MosaiQ
- IBM 양자 하드웨어(`ibm_marrakesh`)에서는 inference만 검증함
- 학습 조건이 달라(위 표) 이 노트북 결과와 직접 비교하지 않음

## 6. 이미지 생성에서의 양자컴퓨팅 - 정리

| 관찰 | 근거 |
|---|---|
| ① 측정 방식이 학습 성패를 가름 | ancilla qubit = 0만 쓰도록 바꾸자 생성 평균과 실제 평균의 상관이 0.00 → 0.49로 오름(8×8 사전 실험). 회로는 그대로임 |
| ② 학습은 시뮬레이터 규모 | 두 모델 모두 exact 상태벡터 시뮬레이션. 실제 하드웨어에서 parameter-shift로 학습하면 MediQ-GAN은 샘플 하나에 회로 1,280회(+ forward 5회) |
| ③ 해상도 한계가 평가 한계 | 16×16에서는 FID를 쓸 수 없어 지표 4종으로 대신함 |
| ④ 큐빗 수에 지수적인 비용 | 16큐빗 상태벡터는 65,536차원으로 7큐빗(128차원)의 512배. 1 step이 로컬 CPU 약 20초로 QGAN(약 2.2초)보다 오래 걸려 수업에서는 fine-tuning만 함 |
| ⑤ 고전 쪽 안정화가 결과를 좌우 | BCE 판별자가 우세함(README "학습 안정성"). 원 논문은 WGAN-GP critic으로 판별자를 제약함 |
| ⑥ 고전 이미지 생성의 우위는 미검증 | 양자 생성모델은 양자 데이터·Hamiltonian 상태 생성에서 먼저 의미를 가짐. 해상도를 높인 사례는 MediQ-GAN처럼 양자회로를 일부만 쓰는 구조에서 나옴 |